---
---
# __Table of Contents__


## __Data Cleaning & Fixing__

### __1. Library Imports__
* __Data Manipulation:__ `pandas` to structure and clean your numbers.

### __2. Outlier Detection & Anomaly Fixes__
1. Working on `train.csv` dataset
2. Working on `test.csv` dataset 
---
---

### __Data Dictionary__

|Variable	     |Definition	     |Key                                             |
|:---------------|:------------------|:-----------------------------------------------|
|passenger_id    |Passenger ID       |
|survival  	     |Survival	         |0 = No, 1 = Yes                                 |
|ticket_class    |Ticket class       |1 = 1st, 2 = 2nd, 3 = 3rd                       |
|name            |Name of person     |
|sex	         |Sex	             |
|age	         |Age in years       |	
|siblings_spouses|# of siblings / spouses aboard the Titanic|
|parents_children|# of parents / children aboard the Titanic|
|ticket	         |Ticket number      |
|fare	         |Passenger fare     |	
|cabin	         |Cabin number       |	
|boarding_port   |Port of Embarkation|	C = Cherbourg, Q = Queenstown, S = Southampton|


### __Variable Notes__

__ticket_class:__ A proxy for socio-economic status (SES)
* 1st = Upper
* 2nd = Middle
* 3rd = Lower

__age:__ Age is fractional if less than 1. If the age is estimated, is it in the form of xx.5

__siblings_spouses:__ The dataset defines family relations in this way...
* Sibling = brother, sister, stepbrother, stepsister
* Spouse = husband, wife (mistresses and fiancés were ignored)

__parents_children:__ The dataset defines family relations in this way...
* Parent = mother, father
* Child = daughter, son, stepdaughter, stepson
* Some children travelled only with a nanny, therefore parch=0 for them.

### __1. Library Imports__
---

In [1]:
# Data manipulation and calculations
import pandas as pd

### __2. Deduplication__
---

### __2.1 Working On `train.csv` Dataset__

#### __Load the Dataset__

In [2]:
# Import the 'train.csv' dataset from 2-under-process folder
train_df = pd.read_csv('../data/2-under-process/train.csv')

In [3]:
# Show the first 5 rows of the dataset
train_df.head()

,passenger_id,survival,ticket_class,name,sex,age,siblings_spouses,parents_children,ticket,fare,cabin,boarding_port
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Get the structural breakdown of the dataset
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   passenger_id      891 non-null    int64  
 1   survival          891 non-null    int64  
 2   ticket_class      891 non-null    int64  
 3   name              891 non-null    object 
 4   sex               891 non-null    object 
 5   age               714 non-null    float64
 6   siblings_spouses  891 non-null    int64  
 7   parents_children  891 non-null    int64  
 8   ticket            891 non-null    object 
 9   fare              891 non-null    float64
 10  cabin             204 non-null    object 
 11  boarding_port     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [5]:
# Create a targeted summary of the boundaries for 'age' and 'fare' columns
boundary_summary = train_df[['age', 'fare']].describe().loc[['min', 'max']]
print("--- Column Boundary Audit ---")
boundary_summary

--- Column Boundary Audit ---


,age,fare
min,0.42,0.0000
max,80.00,512.3292


In [6]:
# Create a loop to profile each requested categorical column cleanly
target_cols = ['sex', 'survived', 'ticket_class', 'boarding_port', 'siblings_spouses', 'parents_children']

print("--- Categorical Distribution Profile ---")
for col in target_cols:
    if col in train_df.columns:
        print(f"\nValue Counts for '{col}' Column:")
        print(train_df[col].value_counts(dropna=False))

--- Categorical Distribution Profile ---

Value Counts for 'sex' Column:
sex
male      577
female    314
Name: count, dtype: int64

Value Counts for 'ticket_class' Column:
ticket_class
3    491
1    216
2    184
Name: count, dtype: int64

Value Counts for 'boarding_port' Column:
boarding_port
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64

Value Counts for 'siblings_spouses' Column:
siblings_spouses
0    608
1    209
2     28
4     18
3     16
8      7
5      5
Name: count, dtype: int64

Value Counts for 'parents_children' Column:
parents_children
0    678
1    118
2     80
5      5
3      5
4      4
6      1
Name: count, dtype: int64


In [7]:
# Profile 'cabin' separately because it has too many unique values to read normally
print("\n--- Cabin Column Overview ---")
print(f"Total missing cabins (NaN): {train_df['cabin'].isnull().sum()} out of {len(train_df)}")
print(f"Number of unique cabin configurations: {train_df['cabin'].nunique()}")
print("Cabin's Categories:\n", train_df['cabin'].value_counts())


--- Cabin Column Overview ---
Total missing cabins (NaN): 687 out of 891
Number of unique cabin configurations: 147
Cabin's Categories:
 cabin
G6             4
C23 C25 C27    4
B96 B98        4
F2             3
D              3
              ..
E17            1
A24            1
C50            1
B42            1
C148           1
Name: count, Length: 147, dtype: int64


In [8]:
# Re-run your verified 'cabin' rule to keep everything in one diagnostic view
cabin_pattern = r'^[A-G]\d{1,3}(?:\s[A-G]\d{1,3})*$'
cabin_anomalies = train_df['cabin'].dropna()[~train_df['cabin'].dropna().str.match(cabin_pattern)]

print(f"--- Cabin Structural Audit ---")
print(f"Total rows violating the Cabin schema: {len(cabin_anomalies)}")
print(cabin_anomalies)

--- Cabin Structural Audit ---
Total rows violating the Cabin schema: 8
75     F G73
128    F E69
292        D
327        D
339        T
473        D
699    F G63
715    F G73
Name: cabin, dtype: object


In [9]:
# 1. Define the production-ready Ticket rule: 
ticket_pattern = r'^\d+$|^[A-Za-z0-9./\s]+ \d+$'

# 2. Find rows where the ticket DOES NOT match our structural rule
ticket_anomalies = train_df[~train_df['ticket'].astype(str).str.match(ticket_pattern)]

# 3. (^[A-Za-z0-9./\s]+ \d+$) matches text pre
print(f"\n\n--- Ticket Structural Audit ---")
print(f"Total rows violating the Ticket schema: {len(ticket_anomalies)}")
if len(ticket_anomalies) > 0:
    print(ticket_anomalies[['name', 'ticket', 'fare']].head())



--- Ticket Structural Audit ---
Total rows violating the Ticket schema: 4
                                name ticket  fare
179              Leonard, Mr. Lionel   LINE   0.0
271     Tornquist, Mr. William Henry   LINE   0.0
302  Johnson, Mr. William Cahoone Jr   LINE   0.0
597              Johnson, Mr. Alfred   LINE   0.0
